# Notebook 12: Capstone — Design a Recommendation System

## Why This Matters

"Design a recommendation system" is the most common ML system design interview question at companies like Netflix, Spotify, Airbnb, LinkedIn, Amazon, and TikTok. It integrates every concept from this series: the design framework, feature stores, training pipelines, online vs batch inference, A/B testing, monitoring, embedding systems, and ML platform design. This capstone walks through a full end-to-end design for an e-commerce product recommendation system—simulating a real 45-minute interview. The goal is to demonstrate how all the components fit together and how to make and justify tradeoffs in a real interview setting.

## 1. Problem Statement and Requirements Clarification

Phase 1 of the framework: clarify requirements before designing anything. We'll work through a realistic e-commerce recommendation scenario.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# Requirements clarification output
requirements = {
    'Problem': 'Design a product recommendation system for an e-commerce platform (Amazon-scale)',
    
    'Functional Requirements': {
        'Primary surfaces': [
            'Homepage: "Recommended for you" — personalized feed',
            'Product detail page: "You might also like" — related items',
            'Cart page: "Frequently bought together" — complementary items',
            'Email: Weekly personalized digest',
        ],
        'Input': 'User ID + surface + context (current page, session history)',
        'Output': 'Ranked list of top-10 product recommendations per surface',
        'Personalization': 'Per-user, based on purchase history + browsing + search',
    },
    
    'Scale Requirements': {
        'DAU': '50 million daily active users',
        'Catalog': '200 million products',
        'Peak QPS': '~35,000 QPS (homepage load peak)',
        'New users/day': '~200,000 (cold start common)',
        'New products/day': '~50,000 (freshness important)',
        'Calculation': '50M users × 20 page loads/day ÷ 86400s × 3x peak ≈ 35K QPS',
    },
    
    'Latency Requirements': {
        'Homepage personalized': '< 100ms p99 (user-facing)',
        'PDP related items': '< 50ms p99 (tab loading)',
        'Email digest': '< 1 hour (batch, pre-computed)',
        'Frequently bought': '< 50ms (typically pre-computed)',
    },
    
    'Freshness Requirements': {
        'User features': 'Recent browsing/purchase: minutes; long-term: hours OK',
        'Item features': 'Price/availability: near-real-time; embedding: daily',
        'Model': 'Retrain weekly (or drift-triggered)',
        'New items': 'Appear in recommendations within hours',
    },
    
    'Non-Functional Requirements': {
        'Privacy': 'GDPR/CCPA compliant; no sharing across users',
        'Diversity': 'Not all recommendations from same brand/category',
        'Freshness': 'Trending/viral items must surface quickly',
        'Safety': 'No offensive/inappropriate products recommended',
        'Availability': '99.99% SLA (revenue-critical surface)',
    },
    
    'Metrics': {
        'Offline': 'NDCG@10, Recall@10 (hold-out future purchases)',
        'Online primary': 'Conversion rate (click → purchase), revenue per session',
        'Online secondary': 'CTR (watch for clickbait), average order value',
        'Guardrails': 'Latency p99, error rate, user complaint rate, category diversity',
        'Long-term': 'Customer lifetime value, return rate',
    }
}

for section, content in requirements.items():
    print(f"\n{'='*65}")
    print(f"  {section}")
    print(f"{'='*65}")
    if isinstance(content, str):
        print(f"  {content}")
    elif isinstance(content, dict):
        for k, v in content.items():
            if isinstance(v, list):
                print(f"  [{k}]")
                for item in v:
                    print(f"    • {item}")
            else:
                print(f"  {k}: {v}")

## 2. High-Level Architecture

Phase 3: Draw the high-level architecture before diving into component details. This establishes the data flow and system boundaries.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(18, 12))
ax.set_xlim(0, 18); ax.set_ylim(0, 12); ax.axis('off')
ax.set_title('E-Commerce Recommendation System: Full Architecture', fontsize=15, fontweight='bold')

def box(ax, x, y, w, h, text, color, fontsize=8.5):
    r = FancyBboxPatch((x,y), w, h, boxstyle="round,pad=0.1",
                       facecolor=color, edgecolor='white', linewidth=1.8, alpha=0.9)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color='white', multialignment='center')

def arr(ax, x1, y1, x2, y2, label='', color='#666'):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    if label:
        ax.text((x1+x2)/2, (y1+y2)/2+0.15, label, fontsize=7, ha='center', color=color)

# === DATA LAYER ===
ax.text(9, 11.5, 'DATA LAYER', ha='center', fontsize=9, color='#888', fontweight='bold')
box(ax, 0.3, 10.2, 3.5, 1.0, 'User Events\n(Kafka)', '#E74C3C')
box(ax, 4.3, 10.2, 3.5, 1.0, 'Product Catalog\n(DB + Kafka)', '#E67E22')
box(ax, 8.3, 10.2, 3.5, 1.0, 'Order/Purchase\nHistory (DWH)', '#F39C12')
box(ax, 12.3, 10.2, 3.5, 1.0, 'Inventory/Price\n(Real-time API)', '#8E44AD')
box(ax, 16.3, 10.2, 1.4, 1.0, 'Content\n(Images/Text)', '#7F8C8D')

# === FEATURE LAYER ===
ax.text(9, 9.5, 'FEATURE LAYER', ha='center', fontsize=9, color='#888', fontweight='bold')
box(ax, 0.3, 8.2, 4.5, 1.0, 'Streaming Features\n(Flink: last-1h browsing, searches)', '#2980B9')
box(ax, 5.5, 8.2, 4.5, 1.0, 'Batch Features\n(Spark: 7d/30d aggregates)', '#3498DB')
box(ax, 11, 8.2, 3.5, 1.0, 'Content Features\n(Image+Text Encoder)', '#1ABC9C')
box(ax, 15, 8.2, 2.7, 1.0, 'Feature Store\n(Feast/Tecton)', '#16A085')

# === MODEL LAYER ===
ax.text(9, 7.4, 'MODEL LAYER', ha='center', fontsize=9, color='#888', fontweight='bold')

# Retrieval
box(ax, 0.3, 6.0, 4, 1.2, 'Two-Tower\nRetrieval Model\n(user + item embeddings)', '#6C3483')
box(ax, 5, 6.0, 4, 1.2, 'ANN Index\n(ScaNN: 200M items)\n→ top-1000 candidates', '#8E44AD')

# Ranking
box(ax, 9.5, 6.0, 4, 1.2, 'Deep Ranking\nModel (GBDT+DNN)\nConverts top-1000 → top-50', '#E74C3C')

# Post-processing
box(ax, 14, 6.0, 3.7, 1.2, 'Post-processing\n(diversity, freshness boost,\nsafety filter)', '#E67E22')

# === SERVING LAYER ===
ax.text(9, 5.2, 'SERVING LAYER', ha='center', fontsize=9, color='#888', fontweight='bold')
box(ax, 0.3, 3.8, 4, 1.2, 'Online Serving\n(REST/gRPC, <100ms)\n35K QPS', '#2C3E50')
box(ax, 5, 3.8, 4, 1.2, 'Batch Scoring\n(Email digest, pre-computed\ncache for stable users)', '#34495E')
box(ax, 9.5, 3.8, 4, 1.2, 'A/B Test Router\n(Traffic allocation\nExperiment tracking)', '#2ECC71')
box(ax, 14, 3.8, 3.7, 1.2, 'Monitoring\n(Drift + metrics\nAlerting)', '#E74C3C')

# === USER ===
box(ax, 7.5, 1.5, 3, 1.0, 'User / Client\n(Homepage, PDP, Cart, Email)', '#7F8C8D')

# Data flow arrows
for x in [2.05, 6.05, 10.05, 14.05]:
    arr(ax, x, 10.2, x, 9.2)

for x in [2.55, 7.75, 12.75]:
    arr(ax, x, 8.2, x, 7.2)

arr(ax, 4.3, 6.6, 5.0, 6.6)
arr(ax, 9.0, 6.6, 9.5, 6.6)
arr(ax, 13.5, 6.6, 14.0, 6.6)

arr(ax, 2.3, 6.0, 2.3, 5.0)
arr(ax, 7.0, 6.0, 7.0, 5.0)
arr(ax, 11.5, 6.0, 11.5, 5.0)
arr(ax, 15.85, 6.0, 15.85, 5.0)

arr(ax, 2.3, 3.8, 4.5, 2.5)
arr(ax, 7.0, 3.8, 7.0, 2.5)
arr(ax, 11.5, 3.8, 10.5, 2.5)

plt.tight_layout()
plt.savefig('recommendation_system_architecture.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Feature Design

Phase 4: Feature design—what signals drive the ranking. Good feature design often matters more than model architecture.

In [ ]:
import pandas as pd

features = pd.DataFrame([
    # User features
    {'Feature': 'user_30d_category_purchases', 'Entity': 'user', 'Type': 'batch', 'Freshness': '24h',
     'Importance': 'High', 'Description': 'Categories user purchased in last 30 days'},
    {'Feature': 'user_7d_browse_categories', 'Entity': 'user', 'Type': 'batch', 'Freshness': '1h',
     'Importance': 'High', 'Description': '7-day browsed category distribution'},
    {'Feature': 'user_session_items', 'Entity': 'user', 'Type': 'streaming', 'Freshness': '5min',
     'Importance': 'High', 'Description': 'Items viewed in current session'},
    {'Feature': 'user_price_sensitivity', 'Entity': 'user', 'Type': 'batch', 'Freshness': '24h',
     'Importance': 'Medium', 'Description': 'Derived from purchase price distribution'},
    {'Feature': 'user_brand_affinity', 'Entity': 'user', 'Type': 'batch', 'Freshness': '24h',
     'Importance': 'High', 'Description': 'Brand preference scores from history'},
    {'Feature': 'user_embedding_256d', 'Entity': 'user', 'Type': 'batch', 'Freshness': 'weekly',
     'Importance': 'High', 'Description': 'Two-tower user embedding from collaborative filtering'},
    
    # Item features
    {'Feature': 'item_7d_click_rate', 'Entity': 'item', 'Type': 'batch', 'Freshness': '1h',
     'Importance': 'High', 'Description': '7-day CTR across all users'},
    {'Feature': 'item_7d_conversion_rate', 'Entity': 'item', 'Type': 'batch', 'Freshness': '1h',
     'Importance': 'High', 'Description': 'Purchase rate conditional on click'},
    {'Feature': 'item_avg_rating', 'Entity': 'item', 'Type': 'batch', 'Freshness': '1h',
     'Importance': 'Medium', 'Description': 'Weighted average customer rating'},
    {'Feature': 'item_price', 'Entity': 'item', 'Type': 'streaming', 'Freshness': 'real-time',
     'Importance': 'High', 'Description': 'Current price (changes with promotions)'},
    {'Feature': 'item_in_stock', 'Entity': 'item', 'Type': 'streaming', 'Freshness': 'real-time',
     'Importance': 'Critical', 'Description': 'Inventory availability flag'},
    {'Feature': 'item_embedding_256d', 'Entity': 'item', 'Type': 'batch', 'Freshness': 'weekly',
     'Importance': 'High', 'Description': 'Two-tower item embedding + content embedding'},
    
    # Cross features (user × item)
    {'Feature': 'user_item_category_match', 'Entity': 'user_item', 'Type': 'on_demand', 'Freshness': 'request-time',
     'Importance': 'High', 'Description': 'Whether item category matches user recent interest'},
    {'Feature': 'user_item_brand_affinity', 'Entity': 'user_item', 'Type': 'on_demand', 'Freshness': 'request-time',
     'Importance': 'High', 'Description': 'User affinity score for item brand'},
    {'Feature': 'embedding_dot_product', 'Entity': 'user_item', 'Type': 'on_demand', 'Freshness': 'request-time',
     'Importance': 'High', 'Description': 'Cosine sim of user and item embeddings'},
    {'Feature': 'price_in_user_budget', 'Entity': 'user_item', 'Type': 'on_demand', 'Freshness': 'request-time',
     'Importance': 'Medium', 'Description': 'Whether price is within user historical range'},
])

print("Feature Design Overview:")
print(f"Total features: {len(features)}")
print("\nBy entity:")
print(features.groupby('Entity')['Feature'].count().to_string())
print("\nBy type:")
print(features.groupby('Type')['Feature'].count().to_string())
print("\nHigh-importance features:")
high_imp = features[features['Importance'] == 'High'][['Feature', 'Entity', 'Type', 'Freshness']]
print(high_imp.to_string(index=False))

print("\nCritical feature: item_in_stock — must be real-time to avoid showing OOS items")
print("Without this, users click OOS items → frustration → lower conversion")

## 4. Model Architecture and Training

Phase 5: Model design. The two-stage architecture (retrieval → ranking) is the standard for large-scale recommendation systems.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

model_design = {
    'Stage 1: Retrieval (Two-Tower Model)': {
        'goal': 'Reduce 200M items to ~1000 candidates in <10ms',
        'architecture': [
            'User tower: [user_embedding, 7d_browse_categories, session_items] → DNN → 256-dim',
            'Item tower: [item_embedding, content_features, popularity] → DNN → 256-dim',
            'Similarity: dot product (ANN index)',
        ],
        'training': [
            'Objective: maximize dot product of purchased user-item pairs vs negatives',
            'Negative sampling: in-batch negatives + hard negatives',
            'Train data: 6 months of purchase events (~5B examples)',
        ],
        'serving': [
            'Item embeddings: pre-computed daily (200M × 256d → 200GB)',
            'ANN index: ScaNN on 64 A100 nodes, 5ms p99',
            'User embedding: computed online (user tower forward pass)',
        ],
        'offline_metric': 'Recall@100: targets 0.85+',
    },
    'Stage 2: Ranking (Deep Neural Network)': {
        'goal': 'Re-rank 1000 candidates to top-50 in <30ms',
        'architecture': [
            'Input: user features (16) + item features (12) + cross features (8) = 36 features',
            'Embedding layers for categorical features (user_id, item_id, category)',
            'DNN: 3 hidden layers [512, 256, 128], BatchNorm, ReLU',
            'Output: probability of purchase (sigmoid)',
        ],
        'training': [
            'Objective: binary cross-entropy on click-to-purchase labels',
            'Training data: 30 days of impression logs (~2B rows)',
            'Features: joined from feature store at impression time',
            'Retrain: weekly on most recent 30 days',
        ],
        'serving': [
            'Online: batch 1000 candidates through ranker (<30ms on GPU)',
            'Features retrieved from online store at request time',
            'Model served via TorchServe with GPU batching',
        ],
        'offline_metric': 'AUC: targets 0.81+, NDCG@10: targets 0.45+',
    },
    'Stage 3: Post-Processing': {
        'goal': 'Apply business rules and diversity to top-50 → top-10',
        'rules': [
            'No out-of-stock items (real-time inventory check)',
            'Max 2 items from same brand in top-10',
            'Max 3 items from same category in top-10',
            'Freshness boost: multiply score by 1.1 for items <7 days old',
            'Remove items user purchased in last 30 days',
            'Safety filter: remove flagged/inappropriate items',
        ],
        'offline_metric': 'Diversity@10, novelty score',
    },
}

for stage, details in model_design.items():
    print(f"\n{'='*70}")
    print(f"  {stage}")
    print(f"  Goal: {details['goal']}")
    print(f"{'='*70}")
    for section, items in details.items():
        if section in ['goal']:
            continue
        if isinstance(items, list):
            print(f"  [{section.title()}]")
            for item in items:
                print(f"    • {item}")
        else:
            print(f"  {section}: {items}")

## 5. Scale Analysis and Tradeoffs

Phase 6: Scaling decisions and tradeoffs. This is what separates senior candidates who think about systems from junior candidates who only think about models.

In [ ]:
import numpy as np

# Scale calculations
print("=" * 65)
print("SCALE CALCULATIONS")
print("=" * 65)

dau = 50_000_000
page_loads_per_user = 20
peak_factor = 3
seconds_per_day = 86_400

avg_qps = dau * page_loads_per_user / seconds_per_day
peak_qps = avg_qps * peak_factor
print(f"\nQPS:")
print(f"  Average QPS: {avg_qps:,.0f}")
print(f"  Peak QPS:    {peak_qps:,.0f} (3× peak factor)")

# Item embedding storage
n_items = 200_000_000
embed_dim = 256
bytes_per = 2  # float16
item_emb_bytes = n_items * embed_dim * bytes_per
print(f"\nItem Embeddings (200M × 256d × float16):")
print(f"  Storage: {item_emb_bytes/1e9:.0f} GB")
print(f"  Doesn't fit in single machine memory (512GB limit)")
print(f"  → Distributed ANN index across ~{int(item_emb_bytes/500e9)+1} × 500GB nodes")

# Training data
impressions_per_day = 1_000_000_000  # 1B impressions/day
training_days = 30
bytes_per_row = 300
training_data_tb = impressions_per_day * training_days * bytes_per_row / 1e12
print(f"\nTraining Data (30 days × 1B impressions):")
print(f"  Rows: {impressions_per_day * training_days / 1e9:.0f}B")
print(f"  Size: {training_data_tb:.1f} TB")
print(f"  → Need Spark distributed training, not single-machine")

# Serving infrastructure
latency_budget_ms = 100
model_inference_ms = 30  # ranking model
feature_retrieval_ms = 15  # feature store lookup
ann_lookup_ms = 8
network_ms = 10
buffer_ms = latency_budget_ms - model_inference_ms - feature_retrieval_ms - ann_lookup_ms - network_ms

print(f"\nLatency Budget (100ms SLA):")
print(f"  ANN lookup:          {ann_lookup_ms}ms")
print(f"  Feature retrieval:   {feature_retrieval_ms}ms")
print(f"  Model inference:     {model_inference_ms}ms")
print(f"  Network overhead:    {network_ms}ms")
print(f"  Buffer remaining:    {buffer_ms}ms")

print("\n" + "=" * 65)
print("KEY TRADEOFFS")
print("=" * 65)

tradeoffs = [
    ('CTR vs Conversion', 'Optimizing CTR alone leads to clickbait. Primary metric: conversion rate.',
     'Use conversion as primary loss; CTR as secondary metric'),
    ('Personalization vs Diversity', 'High personalization → filter bubble / boredom',
     'Diversity re-ranking: max 2 items per brand in top-10'),
    ('Freshness vs Quality', 'New items have no engagement signal',
     'Content-based cold start + exploration budget (5% traffic)'),
    ('Recall vs Precision', 'Two-tower recall@100 matters more than precision',
     'Retrieval: optimize recall; ranking: optimize precision'),
    ('Latency vs Quality', '100ms budget limits model size',
     'Two-stage: cheap retrieval, expensive ranking (on 1000 not 200M)'),
    ('Online vs Batch', 'Batch is cheaper but stale; online is fresh but expensive',
     'Hybrid: batch for stable users, online for all; pre-compute email'),
    ('Retraining cost vs Freshness', 'More frequent retraining costs more',
     'Weekly retrain; drift-triggered if PSI > 0.2'),
]

for tradeoff, tension, resolution in tradeoffs:
    print(f"\n  [{tradeoff}]")
    print(f"  Tension:    {tension}")
    print(f"  Resolution: {resolution}")

## 6. End-to-End Request Flow

Walking through a single request end-to-end demonstrates mastery of how all components interact.

In [ ]:
import time

class RecommendationPipeline:
    """Simplified simulation of the recommendation pipeline for one request."""
    
    def __init__(self):
        self.latency_log = []
    
    def handle_request(self, user_id: str, surface: str, context: dict) -> dict:
        total_start = time.perf_counter()
        self.latency_log = []
        
        # Step 1: Feature retrieval
        t0 = time.perf_counter()
        user_features = self._get_user_features(user_id)
        self.latency_log.append(('Feature retrieval (online store)', (time.perf_counter() - t0) * 1000))
        
        # Step 2: User embedding (user tower forward pass)
        t0 = time.perf_counter()
        user_embedding = self._compute_user_embedding(user_features)
        self.latency_log.append(('User embedding (user tower)', (time.perf_counter() - t0) * 1000))
        
        # Step 3: ANN retrieval
        t0 = time.perf_counter()
        candidates = self._ann_retrieve(user_embedding, k=1000)
        self.latency_log.append(('ANN retrieval (ScaNN)', (time.perf_counter() - t0) * 1000))
        
        # Step 4: Feature assembly for candidates
        t0 = time.perf_counter()
        candidate_features = self._get_candidate_features(candidates, user_features)
        self.latency_log.append(('Candidate feature assembly', (time.perf_counter() - t0) * 1000))
        
        # Step 5: Ranking model
        t0 = time.perf_counter()
        scores = self._rank_candidates(candidate_features)
        self.latency_log.append(('Ranking model (DNN)', (time.perf_counter() - t0) * 1000))
        
        # Step 6: Post-processing
        t0 = time.perf_counter()
        final_recs = self._post_process(candidates, scores, user_features)
        self.latency_log.append(('Post-processing (diversity, filters)', (time.perf_counter() - t0) * 1000))
        
        total_ms = (time.perf_counter() - total_start) * 1000
        
        return {
            'user_id': user_id,
            'surface': surface,
            'recommendations': final_recs[:10],
            'total_latency_ms': total_ms,
            'latency_breakdown': self.latency_log,
        }
    
    def _get_user_features(self, user_id):
        import time; time.sleep(0.001)  # Simulating 1ms Redis lookup
        return {'user_id': user_id, 'categories': ['electronics', 'books'],
                'embedding': __import__('numpy').random.normal(0, 1, 256)}
    
    def _compute_user_embedding(self, features):
        import time; time.sleep(0.0005)  # Simulating 0.5ms forward pass
        return features['embedding']
    
    def _ann_retrieve(self, embedding, k=1000):
        import time; time.sleep(0.005)  # Simulating 5ms ANN
        return [f'item_{i}' for i in range(k)]
    
    def _get_candidate_features(self, candidates, user_features):
        import time; time.sleep(0.008)  # Simulating 8ms batch feature retrieval
        return [{'item_id': c, 'score': 0.0} for c in candidates]
    
    def _rank_candidates(self, features):
        import time; time.sleep(0.015)  # Simulating 15ms ranking inference
        import numpy as np
        return np.random.random(len(features))
    
    def _post_process(self, candidates, scores, user_features):
        import time; time.sleep(0.0005)  # Simulating 0.5ms post-processing
        import numpy as np
        top_idx = np.argsort(scores)[-50:][::-1]
        return [candidates[i] for i in top_idx[:10]]


pipeline = RecommendationPipeline()

result = pipeline.handle_request(
    user_id='user_12345678',
    surface='homepage',
    context={'device': 'mobile', 'country': 'US'}
)

print("Request execution trace:")
print("=" * 60)
print(f"User: {result['user_id']}, Surface: {result['surface']}")
print(f"Total latency: {result['total_latency_ms']:.2f}ms")
print(f"\nLatency breakdown (simulated):")

total_budget = 100
for step, ms in result['latency_breakdown']:
    bar = '█' * max(1, int(ms * 2))
    print(f"  {step:40s}: {ms:6.2f}ms {bar}")

actual_sum = sum(ms for _, ms in result['latency_breakdown'])
print(f"\n  {'Total component time':40s}: {actual_sum:.2f}ms")
print(f"  {'Budget (p99 SLA)':40s}: {total_budget}ms")
print(f"\nTop 5 recommendations: {result['recommendations'][:5]}")

print("\n" + "=" * 60)
print("Production numbers (estimated):")
production_latencies = [
    ('Feature retrieval (Redis batch)', 15),
    ('User embedding (GPU)', 5),
    ('ANN retrieval (ScaNN)', 8),
    ('Candidate features (Redis)', 12),
    ('Ranking DNN (batch 1000)', 25),
    ('Post-processing', 2),
    ('Network + serialization', 10),
    ('Buffer (p99 tail)', 23),
]
total = sum(v for _, v in production_latencies)
for step, ms in production_latencies:
    print(f"  {step:40s}: {ms:3d}ms")
print(f"  {'Total p99':40s}: {total:3d}ms")

## 7. Interview Checklist: Did You Cover Everything?

Use this checklist to evaluate your answer in a mock interview or review session.

In [ ]:
interview_checklist = {
    'Requirements (5 min)': [
        ('✓', 'Clarified scale (DAU, QPS, catalog size)'),
        ('✓', 'Defined latency SLA per surface'),
        ('✓', 'Discussed freshness requirements'),
        ('✓', 'Identified cold start scenarios'),
        ('✓', 'Noted non-functional requirements (diversity, safety)'),
    ],
    'Metrics (3 min)': [
        ('✓', 'Offline: NDCG@10, Recall@100 for retrieval'),
        ('✓', 'Online: conversion rate (not just CTR!)'),
        ('✓', 'Guardrail: latency, complaint rate, diversity'),
        ('⚠', 'Long-term: CLV (easy to forget)'),
    ],
    'Architecture (10 min)': [
        ('✓', 'Two-stage: retrieval → ranking (justified by scale)'),
        ('✓', 'Feature store for online/offline features'),
        ('✓', 'ANN index for retrieval (ScaNN/Faiss)'),
        ('✓', 'Online serving + batch pre-computation for email'),
        ('✓', 'Mentioned hybrid: real-time for user, batch for items'),
    ],
    'Data / Features (8 min)': [
        ('✓', 'User signals: history, session, demographics'),
        ('✓', 'Item signals: popularity, embedding, real-time inventory'),
        ('✓', 'Cross features: user-item affinity, price match'),
        ('✓', 'Real-time inventory as critical feature'),
        ('✓', 'Streaming features for session-level signals'),
    ],
    'Modeling (8 min)': [
        ('✓', 'Two-tower for retrieval (embedding dot product)'),
        ('✓', 'DNN for ranking (cross features enable personalization)'),
        ('✓', 'Post-processing for diversity and business rules'),
        ('✓', 'Cold start: content encoder + exploration policy'),
        ('⚠', 'GBM as alternative to DNN for ranking (gradient boosted)'),
    ],
    'Scale / Trade-offs (9 min)': [
        ('✓', 'Back-of-envelope QPS and storage calculations'),
        ('✓', 'Latency budget decomposition (100ms = sum of steps)'),
        ('✓', 'CTR vs conversion tradeoff (avoid clickbait optimization)'),
        ('✓', 'Personalization vs diversity tradeoff'),
        ('✓', 'Freshness vs quality tradeoff for new items'),
        ('✓', 'A/B testing plan for deploying new model'),
        ('✓', 'Monitoring: drift detection, business metric alerts'),
    ],
}

total_checks = 0
total_items = 0

for phase, items in interview_checklist.items():
    print(f"\n[{phase}]")
    phase_ok = sum(1 for icon, _ in items if icon == '✓')
    total_checks += phase_ok
    total_items += len(items)
    for icon, item in items:
        print(f"  {icon} {item}")
    print(f"  → {phase_ok}/{len(items)} covered")

print(f"\nOVERALL: {total_checks}/{total_items} = {total_checks/total_items:.0%} coverage")
print("\nScoring guide:")
print("  >90% = Strong Hire (L5+ / Senior MLE / Staff level)")
print("  75-90% = Hire (L4 / Senior MLE)")
print("  60-75% = Marginal (MLE II, needs coaching)")
print("  <60% = No hire (missing key system thinking)")

## Summary

| Phase | Key Output | Common Mistake |
|-------|-----------|----------------|
| Requirements | Scale numbers, latency SLA, cold start scope | Assuming rather than clarifying |
| Metrics | Conversion rate primary; CTR secondary; diversity guardrail | Optimizing CTR alone (clickbait) |
| Architecture | Two-stage (retrieval → ranking → post-process) | Jumping to one-stage or over-complex design |
| Features | User + item + cross features; real-time inventory | Missing real-time inventory (OOS problem) |
| Modeling | Two-tower + DNN ranker + diversity post-processing | No cold start strategy |
| Scale | QPS=35K, 200GB embeddings, distributed ANN | No back-of-envelope estimates |
| Tradeoffs | CTR vs conversion, personalization vs diversity | Presenting one design as "the answer" |
| Deployment | A/B test with canary; monitor conversion + drift | No A/B testing or monitoring plan |

**The Series in One Paragraph**

ML system design interviews test your ability to apply a structured framework (Notebook 01) to choose the right feature computation strategy using a feature store (02) driven by a reliable training pipeline (03), served with appropriate architecture (04) using online or batch inference (05), validated by A/B tests (06), monitored for drift and degradation (07, 08), running on a well-designed ML platform (09), powered by embeddings with ANN search (10), with LLM-specific considerations when relevant (11). The capstone (12) shows how all these pieces fit together to answer the canonical interview question.